In [ ]:
cafe_raw_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .load("/Volumes/dev/spark_db/datasets/mini-projects/raw_data/dirty_cafe_sales.csv")
)
cafe_raw_df.display()

In [ ]:
cafe_raw_df = cafe_raw_df.withColumnsRenamed({
    "Transaction ID":"transaction_id",
    "Item":"items",
    "Quantity":"quantity",
    "Price Per Unit":"price_per_unit",
    "Total Spent":"total_spent",
    "Payment Method":"payment_method",
    "Location":"location",
    "Transaction Date":"transaction_date"
})
cafe_raw_df.display()

In [ ]:
from pyspark.sql.types import StringType, LongType, IntegerType, DateType, DoubleType, DecimalType, StructType, StructField

cafe_schema = StructType([
    StructField("transaction_id", StringType()),
    StructField("items", StringType()),
    StructField("quantity", LongType()),
    StructField("price_per_unit", DecimalType()),
    StructField("total_spent", DecimalType()),
    StructField("payment_method", StringType()),
    StructField("location", StringType()),
    StructField("transaction_date", DateType())
])


In [ ]:
cafe_raw_df = (
    spark.read
    .format("csv")
    .option("dateformat","yyyy-MM-dd")
    .option("header",True)
    .schema(cafe_schema)
    .load("/Volumes/dev/spark_db/datasets/mini-projects/raw_data/dirty_cafe_sales.csv")
)
cafe_raw_df.display()

In [ ]:
cafe_raw_df.write.mode("overwrite").saveAsTable("dev.mini_projects.cafe_bronze")

In [ ]:
cafe_raw_df.createOrReplaceTempView("cafe_raw")

In [ ]:
%sql

select count(*), count(distinct transaction_id) from cafe_raw 




In [ ]:
cafe_raw_df.select("transaction_id").distinct().count()

In [ ]:
cafe_raw_df.select("transaction_id").count()

In [ ]:
print(cafe_raw_df.select("transaction_id").distinct().count())
print(cafe_raw_df.select("transaction_id").count())


In [ ]:
%sql
select count(*) from cafe_raw
where quantity is null or  
price_per_unit is null 

In [ ]:
from pyspark.sql.functions import col 
cafe_raw_df.filter(col("quantity").isNull() | col("price_per_unit").isNull()).count()

In [ ]:

cafe_raw_df.filter(col("total_spent").isNull()).count()

In [ ]:
cafe_raw_df.filter((col("total_spent").isNull()) & (col("quantity").isNotNull() & col("price_per_unit").isNotNull())).count()

In [ ]:
%sql 
select count(*) from cafe_raw
where total_spent is null and 
quantity is not null and 
price_per_unit is not null


In [ ]:
%sql

select * from cafe_raw
where total_spent is null and 
(quantity is null or price_per_unit is null)

In [ ]:
cafe_silver_df = cafe_raw_df.filter(~((col("total_spent").isNull()) & ((col("quantity").isNull()) | col("price_per_unit").isNull())))

In [ ]:
cafe_silver_df.display()

In [ ]:
cafe_silver_df.createOrReplaceTempView("cafe_silver")

In [ ]:
%sql

create or replace table  dev.mini_projects.cafe_silver as 
(
with x as (select transaction_id,
items,
quantity,
price_per_unit,
total_spent,
coalesce(total_spent, quantity * price_per_unit) as revenue,
payment_method,
location,
transaction_date
from cafe_silver),

y as (
select * except(items, payment_method, location),
case when items in ("ERROR","UNKNOWN") then null else items end as items,
case when payment_method in ("ERROR","UNKNOWN") then null else payment_method end as payment_method,
case when location in ("ERROR","UNKNOWN") then null else location end as location from x )

select * from y 
)

In [ ]:
%sql 

select * from dev.mini_projects.cafe_silver

In [ ]:
cafe_silver_df = spark.table("dev.mini_projects.cafe_silver")

In [ ]:
cafe_silver_df.display()